In [13]:
import jax
import jax.numpy as jnp
from jax import jit, grad
from functools import partial
import jax.scipy.linalg
import numpy as np
from scipy.optimize import curve_fit

# ==========================================
# 1. CORE JAX ENGINE
# ==========================================

def get_generators():
    gens = []
    gens.append(jnp.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, -1j, 0], [1j, 0, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 1], [0, 0, 0], [1, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, -1j], [0, 0, 0], [1j, 0, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, 1], [0, 1, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[0, 0, 0], [0, 0, -1j], [0, 1j, 0]], dtype=jnp.complex64) * 0.5)
    gens.append(jnp.array([[1, 0, 0], [0, 1, 0], [0, 0, -2]], dtype=jnp.complex64) * (0.5/jnp.sqrt(3)))
    return jnp.stack(gens)

GENERATORS = get_generators()

@partial(jit, static_argnums=(1, 2, 3))
def wilson_loop_trace(U_field, origin, R, T):
    L = U_field.shape[0]
    x, y, z, t = origin
    prod = jnp.eye(3, dtype=jnp.complex64)

    # Trace the loop
    for r in range(R): prod = prod @ U_field[(x+r)%L, y, z, t, 0]
    for r in range(T): prod = prod @ U_field[(x+R)%L, (y+r)%L, z, t, 1]
    for r in range(R): prod = prod @ jnp.conjugate(jnp.swapaxes(U_field[(x+R-1-r)%L, (y+T)%L, z, t, 0], -1, -2))
    for r in range(T): prod = prod @ jnp.conjugate(jnp.swapaxes(U_field[x, (y+T-1-r)%L, z, t, 1], -1, -2))

    return jnp.real(jnp.trace(prod)) / 3.0

@jit
def wilson_action(U_field, beta):
    total_trace = 0.0
    for mu in range(4):
        for nu in range(mu+1, 4):
            U_mu = U_field[:,:,:,:,mu]
            U_nu_shift_mu = jnp.roll(U_field[:,:,:,:,nu], -1, axis=mu)
            U_mu_shift_nu = jnp.roll(U_field[:,:,:,:,mu], -1, axis=nu)
            U_nu = U_field[:,:,:,:,nu]

            U_mu_dag = jnp.conjugate(jnp.swapaxes(U_mu_shift_nu, -1, -2))
            U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))

            P = U_mu @ U_nu_shift_mu @ U_mu_dag @ U_nu_dag
            total_trace += jnp.sum(jnp.real(jnp.trace(P, axis1=-2, axis2=-1)))
    return - (beta / 3.0) * total_trace

@jit
def langevin_step(U_field, key, epsilon, beta):
    # 1. Drift
    dS_dU = grad(wilson_action, argnums=0)(U_field, beta)
    U_dag = jnp.conjugate(jnp.swapaxes(U_field, -1, -2))
    force_raw = U_dag @ dS_dU
    force_ah = (force_raw - jnp.conjugate(jnp.swapaxes(force_raw, -1, -2))) / 2.0
    trace_val = jnp.trace(force_ah, axis1=-2, axis2=-1)[..., None, None]
    force_traceless = force_ah - (trace_val / 3.0) * jnp.eye(3)
    drift_term = -1.0 * force_traceless

    # 2. Noise
    noise_raw = jax.random.normal(key, U_field.shape + (2,))
    noise_c = noise_raw[...,0] + 1j * noise_raw[...,1]
    noise_ah = (noise_c - jnp.conjugate(jnp.swapaxes(noise_c, -1, -2))) / 2.0
    trace_noise = jnp.trace(noise_ah, axis1=-2, axis2=-1)[..., None, None]
    noise_traceless = noise_ah - (trace_noise / 3.0) * jnp.eye(3)

    # 3. Update
    exponent = (epsilon * drift_term) + (jnp.sqrt(epsilon) * noise_traceless)
    update_matrix = jax.scipy.linalg.expm(exponent)
    return update_matrix @ U_field

# ==========================================
# 2. AUTOCORRELATION ENGINE
# ==========================================

def compute_integrated_autocorr(series):
    """
    Computes integrated autocorrelation time (tau_int).
    This is the standard method to measure Critical Slowing Down.
    """
    n = len(series)
    if n == 0: return 0

    mean = np.mean(series)
    c0 = np.var(series)
    if c0 == 0: return 0.0

    tau = 0.5
    # Sum correlation function until it gets noisy
    for t in range(1, n // 50):
        # Standard estimator for autocorrelation function rho(t)
        ct = np.mean((series[:-t] - mean) * (series[t:] - mean))
        rho = ct / c0
        if rho <= 0: break # Stop if correlation vanishes
        tau += rho
    return tau

# ==========================================
# 3. MAIN DEEP SURVEY
# ==========================================

def run_deep_survey():
    print("Initializing T13 'Deep Survey' (20,000 Steps)...")
    print("This will distinguish between Polynomial (LSI) and Exponential (Instanton) scaling.")

    L = 8
    # We scan a wider range to see the curve bend
    betas = [5.6, 5.8, 6.0, 6.2, 6.4]
    epsilon = 0.02

    burn_in = 1000       # Longer burn-in for accuracy
    run_length = 20000   # 10x Statistics vs previous run

    U_field = jnp.broadcast_to(jnp.eye(3, dtype=jnp.complex64), (L,L,L,L,4,3,3))
    key = jax.random.PRNGKey(2024)

    results_beta = []
    results_tau = []

    print("\n" + "="*80)
    print(f"{'Beta':<10} | {'<W>':<10} | {'Tau (Autocorr)':<15} | {'Spectral Gap (1/Tau)'}")
    print("="*80)

    for beta in betas:
        # 1. Thermalize
        for i in range(burn_in):
            key, subkey = jax.random.split(key)
            U_field = langevin_step(U_field, subkey, epsilon, beta)

        # 2. Deep Measurement
        w_history = []
        # We use a JAX loop (scan) for speed here instead of Python loop
        # But for simplicity/visibility, we keep the Python loop structure
        # and print status occasionally.

        for i in range(run_length):
            key, subkey = jax.random.split(key)
            U_field = langevin_step(U_field, subkey, epsilon, beta)

            # Measure 2x2 loop every step
            w = wilson_loop_trace(U_field, (0,0,0,0), 2, 2)
            w_history.append(float(w))

        # 3. Analyze Time Series
        w_history = np.array(w_history)
        tau = compute_integrated_autocorr(w_history)
        gap_est = 1.0 / tau if tau > 0 else 0

        results_beta.append(beta)
        results_tau.append(tau)

        print(f"{beta:<10.1f} | {np.mean(w_history):<10.4f} | {tau:<15.4f} | {gap_est:<10.4f}")

    print("="*80)

    # --- FITTING THE VERDICT ---
    print("\n>>> SCIENTIFIC VERDICT <<<")

    # Model A: Convexity / Polynomial (Standard LSI)
    # Tau ~ Beta^p
    def model_poly(b, a, p): return a * (b**p)

    # Model B: Tunneling / Exponential (Topological Gap)
    # Tau ~ exp(c * sqrt(Beta)) -- Scaling of Instanton Action
    def model_exp(b, a, c): return a * np.exp(c * np.sqrt(b))

    try:
        popt_poly, _ = curve_fit(model_poly, results_beta, results_tau, p0=[0.1, 2.0], maxfev=10000)
        popt_exp, _ = curve_fit(model_exp, results_beta, results_tau, p0=[0.01, 3.0], maxfev=10000)

        res_poly = np.sum((np.array(results_tau) - model_poly(np.array(results_beta), *popt_poly))**2)
        res_exp = np.sum((np.array(results_tau) - model_exp(np.array(results_beta), *popt_exp))**2)

        print(f"Polynomial Fit Error (MSE): {res_poly:.5f}")
        print(f"Exponential Fit Error (MSE): {res_exp:.5f}")

        if res_exp < res_poly:
            print("\nRESULT: The data fits the EXPONENTIAL Model best.")
            print("INTERPRETATION: The Mass Gap is driven by Instantons (Topology).")
            print("ACTION: The 'Convexity' proof strategy must be abandoned for 'Topological LSI'.")
        else:
            print("\nRESULT: The data fits the POLYNOMIAL Model best.")
            print("INTERPRETATION: The Mass Gap acts like a Convex Potential.")
            print("ACTION: The 'Convexity' proof strategy is VIABLE.")

    except Exception as e:
        print(f"Fitting error: {e}")

if __name__ == "__main__":
    run_deep_survey()

Initializing T13 'Deep Survey' (20,000 Steps)...
This will distinguish between Polynomial (LSI) and Exponential (Instanton) scaling.

Beta       | <W>        | Tau (Autocorr)  | Spectral Gap (1/Tau)
5.6        | 0.1816     | 3.6241          | 0.2759    
5.8        | 0.1822     | 4.0885          | 0.2446    
6.0        | 0.1928     | 4.0005          | 0.2500    
6.2        | 0.2114     | 4.2925          | 0.2330    
6.4        | 0.2069     | 3.6016          | 0.2777    

>>> SCIENTIFIC VERDICT <<<
Polynomial Fit Error (MSE): 0.35911
Exponential Fit Error (MSE): 0.35967

RESULT: The data fits the POLYNOMIAL Model best.
INTERPRETATION: The Mass Gap acts like a Convex Potential.
ACTION: The 'Convexity' proof strategy is VIABLE.
